In [ ]:
%%sql -r dataframe_1
use role sysadmin;
use warehouse compute_wh;

In [ ]:
%%sql -r dataframe_2
use role accountadmin;
grant execute task,execute managed task on account to role sysadmin;

In [ ]:
%%sql -r dataframe_3
use role sysadmin;

****
### Create databsae and schema
****

In [ ]:
%%sql -r dataframe_5
create or replace database legacy_db
comment = 'this is legacy_db database for stream & task demo';

use database legacy_db;

In [ ]:
%%sql -r dataframe_6
create or replace schema source_sch
comment = 'this is stage schema in legacy_db database';

create or replace schema raw_sch
comment = 'this is raw schema in legacy_db database';

create or replace schema clean_sch
comment = 'this is clean schema in legacy_db database';

create or replace schema consumption_sch
comment = 'this is consumption schema in legacy_db database';

In [ ]:
%%sql -r dataframe_7
show databases like 'legacy%';

In [ ]:
%%sql -r dataframe_8
show schemas in database legacy_db;

In [ ]:
%%sql -r dataframe_9
use schema legacy_db.source_sch;

****
### Create file format
****

In [ ]:
%%sql -r dataframe_11
create or replace file format legacy_db.source_sch.csv_format
    type = 'csv' 
    compression = 'auto' 
    field_delimiter = ',' 
    record_delimiter = '\n'  
    field_optionally_enclosed_by = '"' 
    skip_header = 1
;

In [ ]:
%%sql -r dataframe_12
show file formats;

In [ ]:
%%sql -r dataframe_13
describe file format csv_format;

****
Create an internal stage location
****

In [ ]:
%%sql -r dataframe_15
create or replace stage legacy_db.source_sch.my_stage
    file_format = (format_name = 'legacy_db.source_sch.csv_format')
    directory = (enable = true auto_refresh = true)
    comment = 'this is snowflake internal stage to stage the data files under the legacy_db/source schema';

In [ ]:
%%sql -r dataframe_16
show stages;

In [ ]:
%%sql -r dataframe_17
describe stage legacy_db.source_sch.my_stage;

****
### Data Loading into Stage
****
Data Loaded from snowsight into stage my_stage

In [ ]:
list @legacy_db.source_sch.my_stage/customer;

In [ ]:
list @legacy_db.source_sch.my_stage/customer/delta/;

In [ ]:
list @legacy_db.source_sch.my_stage/order/;

In [ ]:
list @legacy_db.source_sch.my_stage/order/delta/;

In [ ]:
select
    t.$1,
    t.$2,
    t.$3,
    t.$4,
    t.$5,
    t.$6,
    t.$7,
    current_timestamp(),
    metadata$file_row_number,
    metadata$filename
from @legacy_db.source_sch.my_stage/customer/ as t;

In [ ]:
select
    t.$1,
    t.$2,
    t.$3,
    t.$4,
    t.$5,
    t.$6,
    t.$7,
    t.$8,
    current_timestamp(),
    metadata$file_row_number,
    metadata$filename
from @legacy_db.source_sch.my_stage/order/ as t;

Load bulk data into my_stage from snowsight & check above list stage queries to verify newly added files.